# Combined component and acoustic objectives

Train one objective across five fixed seeds using an existing experiment bundle.


## Configuration

Select input and output paths, device and objective before defining the runtime. The combined model uses the selected finite acoustic coefficient. Training is disabled by default.


In [ ]:
import hashlib
import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root")

CODE_ROOT = find_project_root()
from pathlib import Path

EXPERIMENT_ID = 'duyu-chinese-lessons-zh-cn-v1'
EXPERIMENT_PATH = CODE_ROOT / "datasets" / "experiments" / EXPERIMENT_ID
OUTPUT_ROOT = CODE_ROOT / "runs"

EXPECTED_EXPERIMENT_MANIFEST_SHA256 = '719fc321a121c06a25c5247533564c15ce02f26e6bc82aa20c33fc567a227185'
DEVICE = "cpu"
LAMBDA_VALUE = 1
MODEL_ID = f"components-acoustic-lambda-{LAMBDA_VALUE}"
RUN_REAL_TRAINING = False


### Build provenance

Canonical source hashes and the experiment manifest pin identify this notebook build.


In [ ]:
BUILD_METADATA = {
  "baseline_project_sha": "f7029f129571bc5f2c2546cc86a25a19d3d0aa1f",
  "embedded_source_sha256": "70ec9db354686cdf74195a0c2df47a9a336153540d0d2426b92d0770b26880a6",
  "experiment_id": "duyu-chinese-lessons-zh-cn-v1",
  "experiment_manifest_sha256": "719fc321a121c06a25c5247533564c15ce02f26e6bc82aa20c33fc567a227185",
  "source_sha256": {
    "__init__.py": "bfa7860bfad7236ce5bbd3aec032161527e7cd8e1b6607c79c15f14853b9b6b2",
    "artifacts.py": "9447013a45a347b7df6735cde9dc7c40f0931d53792165999c482c384321d64d",
    "data.py": "2be23fc5b5a5916c66074dc447143008719ee3c4b1eb2192b73d5f8b30be5989",
    "data_contract.py": "5c526086867ed477286c1f369ca78e1ccd44fa52ed8453846d87a664f2971fa3",
    "experiment.py": "518c606af5b1535d928e3bb3a7425a15db77525ea4bc944f2af74ffa0248ddb0",
    "ids.py": "832b20b5f97f8328150a6efd57847d9cc56378b703117da1bde65759f29c37ab",
    "metrics.py": "10d41722970c5472cfe9c3f09346ae977d95dab520fc8d72a4f26d3e7f935fbc",
    "model.py": "34932db3ea2ac8b2bf3e0d2631e563dcbe22d686c3408a0d113cd499ff3630a1",
    "objectives.py": "2cdb95c3044103e6138d28038d696e43b200ea7b0e94c3131cac9a3fd3c0df47",
    "pinyin.py": "6d080de3613cd219edd5553512e17e4fe596a7bb881d59c0f6acd654bb85f1d3",
    "reproducibility.py": "ef59e1eb6453cfb149d11b6a4f2c7227d1aa8a8b7f354fb1a9a9b6e1590296aa",
    "training.py": "e567ed0eef9646479ce3cf50089ea024b1d12946d18e24444cde5dee00e1108d",
    "training_data.py": "6b16cc0d66276f782c9e839ec37fe6dbce703723f079ef706ef1be94a74cb684"
  }
}


## Dependencies and project runtime

Verify canonical source hashes and import the local SDK in a fresh kernel. The SDK handles bundle validation, batching, encoders, objectives, reproducibility, training diagnostics and checkpoints.


In [ ]:
import hashlib
import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root")

CODE_ROOT = find_project_root()
if any(name == "phonetics" or name.startswith("phonetics.") for name in sys.modules):
    raise RuntimeError("Composite training requires a fresh kernel")
for name, expected in BUILD_METADATA["source_sha256"].items():
    actual = hashlib.sha256((CODE_ROOT / "src" / "phonetics" / name).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Project source differs from this notebook: {name}")
sys.path.insert(0, str(CODE_ROOT / "src"))
from phonetics.training import train_experiment


## Training pipeline

1. Validate the experiment and construct length-aware batches.
2. Encode IDS characters and full sentence context, then apply the selected readouts.
3. Optimize the configured objective across five sequential seeds.
4. Save best/last checkpoints, gradient diagnostics and summaries under the selected output root.

The standalone companion displays the implementation as ordinary Python cells.


## Run training

Enable only after reviewing the pinned experiment, device and output path. This cell trains the selected objective for all five seeds.


In [ ]:
if RUN_REAL_TRAINING:
    result = train_experiment(
        EXPERIMENT_PATH, model_id=MODEL_ID, runs_root=OUTPUT_ROOT, device=DEVICE,
        expected_manifest_sha256=EXPECTED_EXPERIMENT_MANIFEST_SHA256,
    )
    result["summary"]
else:
    print("Training is disabled. Review the pinned experiment and settings before enabling it.")
